In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from Bio import SeqIO

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

WindowsPath('c:/Users/Dell - i5 11th Gen/Desktop/atm-protein-conservation-explorer')

In [2]:
variants_file = project_root / "data" / "processed" / "prepared_atm_missense_vus.csv"
variants = pd.read_csv(variants_file)

aligned_fasta = project_root / "data" / "processed" / "atm_representative_proteins_aligned.fasta"
aligned_records = list(SeqIO.parse(aligned_fasta, "fasta"))

required_columns = {"protein_change", "protein_position", "reference_amino_acid", "alternate_amino_acid"}
missing_columns = required_columns - set(variants.columns)
if missing_columns:
    raise ValueError(f"Missing variant columns: {sorted(missing_columns)}")

aligned_lengths = {len(record.seq) for record in aligned_records}
if len(aligned_lengths) != 1:
    raise ValueError("Aligned sequences do not all have the same length.")

if len({record.id for record in aligned_records}) != len(aligned_records):
    raise ValueError("Duplicate FASTA record IDs found.")

human_records = [record for record in aligned_records if record.id == "NP_000042.3"]
if len(human_records) != 1:
    raise ValueError(f"Expected one human ATM record, found {len(human_records)}.")

human_record = human_records[0]
comparison_records = [record for record in aligned_records if record.id != human_record.id]

print("Variant rows:", len(variants))
print("Aligned records:", len(aligned_records))
print("Comparison species:", len(comparison_records))
print("Alignment length:", next(iter(aligned_lengths)))

Variant rows: 4526
Aligned records: 827
Comparison species: 826
Alignment length: 4301


In [3]:
variants = variants.dropna(
    subset=["protein_position", "reference_amino_acid", "alternate_amino_acid"]
).copy()

variants["protein_position"] = pd.to_numeric(variants["protein_position"], errors="raise").astype(int)
variants["reference_amino_acid"] = variants["reference_amino_acid"].astype(str).str.upper()
variants["alternate_amino_acid"] = variants["alternate_amino_acid"].astype(str).str.upper()

human_aligned_sequence = str(human_record.seq).upper()

position_to_alignment_index = {}
human_position = 0
for alignment_index, residue in enumerate(human_aligned_sequence):
    if residue != "-":
        human_position += 1
        position_to_alignment_index[human_position] = alignment_index

variant_positions = sorted(variants["protein_position"].unique())
unmapped_positions = [
    position for position in variant_positions if position not in position_to_alignment_index
]
if unmapped_positions:
    raise ValueError(f"Unmapped human positions: {unmapped_positions[:10]}")

print("Human protein length:", human_position)
print("Unique variant positions:", len(variant_positions))
print("All positions mapped:", len(unmapped_positions) == 0)

Human protein length: 3056
Unique variant positions: 1793
All positions mapped: True


In [4]:
comparison_matrix = np.array(
    [list(str(record.seq).upper()) for record in comparison_records], dtype="U1"
)

excluded_symbols = ["-", "X", "?"]
position_rows = []
residue_counts_by_position = {}

for protein_position in variant_positions:
    alignment_index = position_to_alignment_index[protein_position]
    human_residue = human_aligned_sequence[alignment_index]
    column_residues = comparison_matrix[:, alignment_index]

    usable_residues = column_residues[~np.isin(column_residues, excluded_symbols)]
    unique_residues, residue_counts = np.unique(usable_residues, return_counts=True)
    residue_count_dictionary = dict(zip(unique_residues.tolist(), residue_counts.astype(int).tolist()))
    residue_counts_by_position[protein_position] = residue_count_dictionary

    usable_species_count = len(usable_residues)
    matching_species_count = residue_count_dictionary.get(human_residue, 0)
    conservation_score = (
        matching_species_count / usable_species_count if usable_species_count > 0 else np.nan
    )

    position_rows.append({
        "protein_position": protein_position,
        "alignment_column": alignment_index + 1,
        "human_residue": human_residue,
        "comparison_species_count": len(comparison_records),
        "usable_species_count": usable_species_count,
        "matching_species_count": matching_species_count,
        "gap_species_count": int(np.count_nonzero(column_residues == "-")),
        "ambiguous_species_count": int(np.count_nonzero(np.isin(column_residues, ["X", "?"]))),
        "conservation_score": conservation_score,
        "conservation_percent": conservation_score * 100,
    })

position_scores = pd.DataFrame(position_rows).sort_values("protein_position").reset_index(drop=True)
position_scores.head()

,protein_position,alignment_column,human_residue,comparison_species_count,usable_species_count,matching_species_count,gap_species_count,ambiguous_species_count,conservation_score,conservation_percent
0,2,83,S,826,814,761,12,0,0.934889,93.488943
1,3,84,L,826,818,767,8,0,0.937653,93.765281
2,4,85,V,826,815,125,11,0,0.153374,15.337423
3,5,87,L,826,814,779,12,0,0.957002,95.700246
4,7,89,D,826,815,598,11,0,0.733742,73.374233


In [5]:
variant_scores = variants.merge(
    position_scores, on="protein_position", how="left", validate="many_to_one"
)

variant_scores["reference_matches_human"] = (
    variant_scores["reference_amino_acid"] == variant_scores["human_residue"]
)

variant_scores["alternate_species_count"] = [
    residue_counts_by_position[int(pos)].get(alt, 0)
    for pos, alt in zip(variant_scores["protein_position"], variant_scores["alternate_amino_acid"])
]

variant_scores["alternate_species_fraction"] = (
    variant_scores["alternate_species_count"]
    .div(variant_scores["usable_species_count"])
    .where(variant_scores["usable_species_count"] > 0)
)

variant_scores["alternate_seen_in_species"] = variant_scores["alternate_species_count"] > 0

variant_scores.head()

,Unnamed: 0,gene,clinvar_variation_id,vcv_accession,rs_id,clinvar_variation,transcript,hgvs_c,hgvs_p,protein_change,...,usable_species_count,matching_species_count,gap_species_count,ambiguous_species_count,conservation_score,conservation_percent,reference_matches_human,alternate_species_count,alternate_species_fraction,alternate_seen_in_species
0,0,ATM,4170131,VCV004170131,rs4279232,NM_000051.4(ATM):c.4A>C (p.Ser2Arg),NM_000051.4,c.4A>C,p.Ser2Arg,S2R,...,814,761,12,0,0.934889,93.488943,True,0,0.000000,False
1,1,ATM,640846,VCV000640846,rs639367,NM_000051.4(ATM):c.4A>G (p.Ser2Gly),NM_000051.4,c.4A>G,p.Ser2Gly,S2G,...,814,761,12,0,0.934889,93.488943,True,0,0.000000,False
2,2,ATM,181943,VCV000181943,rs180377,NM_000051.4(ATM):c.5G>C (p.Ser2Thr),NM_000051.4,c.5G>C,p.Ser2Thr,S2T,...,814,761,12,0,0.934889,93.488943,True,6,0.007371,True
3,3,ATM,922075,VCV000922075,rs911284,NM_000051.4(ATM):c.6T>A (p.Ser2Arg),NM_000051.4,c.6T>A,p.Ser2Arg,S2R,...,814,761,12,0,0.934889,93.488943,True,0,0.000000,False
4,4,ATM,1337452,VCV001337452,rs1328461,NM_000051.4(ATM):c.6T>G (p.Ser2Arg),NM_000051.4,c.6T>G,p.Ser2Arg,S2R,...,814,761,12,0,0.934889,93.488943,True,0,0.000000,False


In [6]:
if variant_scores["alignment_column"].isna().any():
    raise ValueError("Some variants were not mapped to alignment columns.")

reference_mismatches = variant_scores[~variant_scores["reference_matches_human"]]
if len(reference_mismatches) > 0:
    display(reference_mismatches[
        ["protein_change", "protein_position", "reference_amino_acid", "human_residue"]
    ].head(20))
    raise ValueError("Some variant reference residues do not match human ATM.")

if not position_scores["conservation_score"].dropna().between(0, 1).all():
    raise ValueError("Some conservation scores are outside 0-1.")

print("Scored variant rows:", len(variant_scores))
print("Scored unique positions:", len(position_scores))
print("Reference mismatches:", len(reference_mismatches))
print("Conservation score range:", position_scores["conservation_score"].min(), "to", position_scores["conservation_score"].max())

variant_scores[[
    "protein_change", "protein_position", "alignment_column", "human_residue",
    "usable_species_count", "matching_species_count", "conservation_score", "alternate_species_count",
]].head(10)

Scored variant rows: 4526
Scored unique positions: 1793
Reference mismatches: 0
Conservation score range: 0.0036363636363636364 to 1.0


,protein_change,protein_position,alignment_column,human_residue,usable_species_count,matching_species_count,conservation_score,alternate_species_count
0,S2R,2,83,S,814,761,0.934889,0
1,S2G,2,83,S,814,761,0.934889,0
2,S2T,2,83,S,814,761,0.934889,6
3,S2R,2,83,S,814,761,0.934889,0
4,S2R,2,83,S,814,761,0.934889,0
5,L3P,3,84,L,818,767,0.937653,1
6,V4L,4,85,V,815,125,0.153374,9
7,L5V,5,87,L,814,779,0.957002,9
8,L5F,5,87,L,814,779,0.957002,3
9,L5R,5,87,L,814,779,0.957002,0


In [7]:
position_output_file = project_root / "data" / "processed" / "position_scores.csv"
variant_output_file = project_root / "data" / "processed" / "variant_scores.csv"

position_scores.to_csv(position_output_file, index=False)
variant_scores.to_csv(variant_output_file, index=False)

print("Position scores saved:", position_output_file)
print("Variant scores saved:", variant_output_file)
print("Position output rows:", len(position_scores))
print("Variant output rows:", len(variant_scores))

Position scores saved: c:\Users\Dell - i5 11th Gen\Desktop\atm-protein-conservation-explorer\data\processed\position_scores.csv
Variant scores saved: c:\Users\Dell - i5 11th Gen\Desktop\atm-protein-conservation-explorer\data\processed\variant_scores.csv
Position output rows: 1793
Variant output rows: 4526
